In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm'])

0

In [2]:
import pandas as pd
import numpy as np
import joblib
import ast
import time # Import the time module
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import SVC # Import Support Vector Classifier
from sklearn.metrics import classification_report
from tqdm.notebook import tqdm # For training progress bar

# 1. Load your data into a Pandas DataFrame from the CSV file
df = pd.read_csv('re_labeled_FIXED_final.csv')

# 2. Define feature columns and create X
feature_columns = [
    'temperature_cleaned',
    'tds_cleaned',
    'turbidity_cleaned',
    'pH_cleaned',
    'Rate of Change (ΔT/Δt)',
    'Moving Average Deviation',
    'Short-Term Gradient (ΔNTU)',
    'Rolling Variance (σ²ₚₕ)',
    'temp_fault',
    'tds_fault',
    'turbidity_fault',
    'ph_fault'
]
X = df[feature_columns].copy()

# Ensure all columns are numeric for mean calculation and handle missing values
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = pd.to_numeric(X[col], errors='coerce')
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].mean())

# 3. Determine mlb and y_binarized
fault_labels_list = df['fault detection'].apply(ast.literal_eval)
mlb = MultiLabelBinarizer()
y_binarized = mlb.fit_transform(fault_labels_list)

# 4. Split the data into training and testing sets
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y_binarized, test_size=0.2, random_state=42
)

print("Starting SVM multi-label classification training...")

# Record start time
start_time = time.time()

# Create a list to hold trained estimators
trained_svm_estimators = []

# Use tqdm for a progress bar over the labels
pbar = tqdm(range(y_train_multi.shape[1]), desc="Training SVM for each label")
for i in pbar:
    label_name = mlb.classes_[i]
    # Create a new SVC classifier for each label
    # Using 'linear' kernel for efficiency with potentially many features
    # probability=True is needed for predict_proba (used for thresholding)
    single_label_svm = SVC(kernel='linear', C=1.0, random_state=42, probability=True)
    single_label_svm.fit(X_train_multi, y_train_multi[:, i])
    trained_svm_estimators.append(single_label_svm)
    # Update postfix with real-time elapsed
    pbar.set_postfix_str(f"Elapsed: {time.time() - start_time:.2f}s")

# Record end time (still useful for internal calculation or if user changes mind)
end_time = time.time()

print("SVM multi-label models trained successfully.")
# Removed: print(f"Time elapsed for SVM training: {end_time - start_time:.2f} seconds")

# 1. Save the model checkpoint
svm_model_filename = 'multi_label_svm_model.joblib'
joblib.dump(trained_svm_estimators, svm_model_filename)
print(f"SVM multi-label model saved successfully to {svm_model_filename}")

# 2. Make predictions on the test set
# Initialize an empty array to store predictions (num_samples, num_labels)
y_pred_svm = np.zeros_like(y_test_multi, dtype=int)

for i, model in enumerate(trained_svm_estimators):
    # Use predict_proba to get probabilities, then threshold at 0.5
    y_pred_svm[:, i] = (model.predict_proba(X_test_multi)[:, 1] > 0.5).astype(int)

# Output multi-label classification report
print("\nMulti-Label Classification Report for SVM:\n")
print(classification_report(y_test_multi, y_pred_svm, target_names=mlb.classes_, zero_division=0))

ModuleNotFoundError: No module named 'tqdm'